# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/praveenadanthapally/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — Click Capture by Position Tier

The paper reports that weighted CTR declines as visibility moves away from the top of search results, with the strongest weighted CTR in the Top 3 and much lower CTR in deeper position tiers.

**Methodology question:** How is the position-based label or grouping defined, and are the position tiers evaluated on independent observations or repeated page observations? Because CTR, clicks, impressions, and average position are measured from the same search-performance data, I would want to understand whether the validation design prevents repeated pages or clients from appearing across comparison groups in a way that could make the relationship look stronger than it is.

This does not challenge the observed pattern. It is a question about whether the validation design supports interpreting the result as a stable decision-support signal rather than a causal effect of moving position.

### Finding 2 — The Freshness Multiplier

The paper reports that mature pages refreshed within 30 days had substantially higher health and impressions than the comparison group, including a measured 3.2x health increase and 57x more impressions in the reported portfolio analysis.

**Methodology question:** Where does the refresh comparison label come from, and does the validation design control for pre-existing differences between refreshed and untouched pages? Pages selected for refresh may already have stronger demand, historical visibility, or strategic importance. I would therefore want to know whether the comparison uses matched pages, a time-aware design, or another method that reduces selection and survivor bias.

The paper itself appropriately narrows this interpretation by describing refresh as a measured lever in this portfolio rather than proof that refreshing any page will produce the same result. I would preserve that cautious interpretation when using the finding as decision-support.


In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Why I am changing the validation split

The Week-5 Random Forest achieved an observed NDCG of **0.4629**, compared with **0.4391** for the Week-4 baseline on the original evaluation setup.

For this validation audit, I reuse the corrected Week-5 model data, features, and `future_ctr` target, but evaluate the modeling approach with a **client-grouped split**.

The purpose of this stricter split is to test whether the ranking approach generalizes to clients that were not used during training. `client_id` is used only to define the groups and is excluded from the predictive feature set.

The final grouped validation contains **23 training clients and 6 test clients**, with **zero client overlap**.

The resulting grouped NDCG is reported as a separate validation measurement. It should not be interpreted as proof that the model will perform identically on future clients or future search data.


In [14]:
# ============================================================
# WEEK 6 — HONEST CLIENT-GROUPED VALIDATION
# Uses the REAL Week-5 model_df and future_ctr target
# ============================================================

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import ndcg_score, mean_absolute_error, mean_squared_error


# ------------------------------------------------------------
# 1. Check that the corrected Week-5 model data exists
# ------------------------------------------------------------

if "model_df" not in globals():
    raise NameError(
        "model_df is not defined. "
        "Run the corrected Week-5 model-data preparation cells first."
    )

if "future_ctr" not in model_df.columns:
    raise ValueError(
        "future_ctr is not present in model_df. "
        "Do NOT create it from clicks_last_30d / impressions_last_30d. "
        "Run the corrected Week-5 target-construction cells first."
    )

print("REAL WEEK-5 MODEL DATA FOUND")
print("Rows:", len(model_df))
print("Columns:", len(model_df.columns))


# ------------------------------------------------------------
# 2. Define the target
# ------------------------------------------------------------

target_col = "future_ctr"

print()
print("Target column:", target_col)


# ------------------------------------------------------------
# 3. Define legitimate Week-5 predictive features
# ------------------------------------------------------------

feature_cols = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "trend_pct"
]


# Keep only features actually available
feature_cols = [
    c for c in feature_cols
    if c in model_df.columns
]


# ------------------------------------------------------------
# 4. Explicit leakage protection
# ------------------------------------------------------------

leakage_cols = [
    "ctr",
    "future_ctr"
]

feature_cols = [
    c for c in feature_cols
    if c not in leakage_cols
]


required_cols = (
    feature_cols
    + [target_col, "client_id"]
)

model_df = model_df.dropna(
    subset=required_cols
).copy()


print()
print("MODEL DATA")
print("Rows:", len(model_df))
print("Features:", len(feature_cols))
print("Target:", target_col)

print()
print("Features used:")
for c in feature_cols:
    print("-", c)


# ------------------------------------------------------------
# 5. Final model matrix
# ------------------------------------------------------------

X = model_df[feature_cols]
y = model_df[target_col]
groups = model_df["client_id"]

print()
print("FINAL MODEL MATRIX")
print("Rows:", len(X))
print("Features:", len(feature_cols))
print("Target:", target_col)


# ------------------------------------------------------------
# 6. Client-grouped validation
# ------------------------------------------------------------

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

train_clients = set(
    model_df.iloc[train_idx]["client_id"]
)

test_clients = set(
    model_df.iloc[test_idx]["client_id"]
)

client_overlap = (
    train_clients & test_clients
)


print()
print("GROUPED VALIDATION")
print("------------------")
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(client_overlap))


# ------------------------------------------------------------
# 7. Train Random Forest only on training clients
# ------------------------------------------------------------

honest_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

honest_model.fit(
    X_train,
    y_train
)

pred = honest_model.predict(X_test)


# ------------------------------------------------------------
# 8. Prediction error metrics
# ------------------------------------------------------------

mae = mean_absolute_error(
    y_test,
    pred
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        pred
    )
)


# ------------------------------------------------------------
# 9. NDCG within each unseen client
# ------------------------------------------------------------

test_eval = model_df.iloc[test_idx][
    ["client_id", target_col]
].copy()

test_eval["prediction"] = pred

client_ndcgs = []

for client_id, client_data in test_eval.groupby(
    "client_id"
):

    # NDCG requires at least two observations
    if len(client_data) < 2:
        continue

    true_values = (
        client_data[target_col]
        .to_numpy()
    )

    predicted_values = (
        client_data["prediction"]
        .to_numpy()
    )

    score = ndcg_score(
        [true_values],
        [predicted_values]
    )

    client_ndcgs.append(score)


if len(client_ndcgs) > 0:
    honest_ndcg = float(
        np.mean(client_ndcgs)
    )
else:
    honest_ndcg = np.nan


# ------------------------------------------------------------
# 10. Final Week-6 result
# ------------------------------------------------------------

print()
print("=" * 60)
print("FINAL WEEK-6 CLIENT-GROUPED VALIDATION")
print("=" * 60)

print(
    "Training clients:",
    len(train_clients)
)

print(
    "Test clients:",
    len(test_clients)
)

print(
    "Client overlap:",
    len(client_overlap)
)

print(
    "Honest grouped NDCG:",
    round(honest_ndcg, 4)
)

print(
    "MAE:",
    round(mae, 4)
)

print(
    "RMSE:",
    round(rmse, 4)
)


# ------------------------------------------------------------
# 11. Comparison with previous results
# ------------------------------------------------------------

comparison = pd.DataFrame({
    "Method": [
        "Week-4 baseline",
        "Week-5 Random Forest",
        "Week-6 Random Forest — client-grouped validation"
    ],
    "NDCG": [
        0.4391,
        0.4629,
        honest_ndcg
    ]
})

print()
print("RESULTS COMPARISON")

display(comparison)

REAL WEEK-5 MODEL DATA FOUND
Rows: 17917
Columns: 45

Target column: future_ctr

MODEL DATA
Rows: 17917
Features: 27
Target: future_ctr

Features used:
- search_volume
- competition
- cpc
- word_count
- char_count
- impressions_90d
- clicks_90d
- pageviews_90d
- sessions_90d
- users_90d
- engaged_sessions_90d
- ai_sessions_90d
- scroll_events_90d
- days_with_impressions
- days_with_sessions
- impressions_last_30d
- clicks_last_30d
- sessions_last_30d
- impressions_prev_30d
- clicks_prev_30d
- sessions_prev_30d
- content_age_days
- days_since_last_update
- engagement_rate
- scroll_rate
- ai_traffic_pct
- trend_pct

FINAL MODEL MATRIX
Rows: 17917
Features: 27
Target: future_ctr

GROUPED VALIDATION
------------------
Training rows: 12493
Test rows: 5424
Training clients: 23
Test clients: 6
Client overlap: 0

FINAL WEEK-6 CLIENT-GROUPED VALIDATION
Training clients: 23
Test clients: 6
Client overlap: 0
Honest grouped NDCG: 0.8157
MAE: 0.0009
RMSE: 0.0053

RESULTS COMPARISON


,Method,NDCG
0,Week-4 baseline,0.439100
1,Week-5 Random Forest,0.462900
2,Week-6 Random Forest — client-grouped validation,0.815744


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Leakage audit

I reviewed the final Week-5 feature set against the target and the data-generation process.

The main checks were:

1. **Target leakage:** no feature should directly contain or encode the target used for ranking.
2. **Future information:** features available only after the prediction/evaluation point should not be used to make the prediction.
3. **Derived-target overlap:** features that are components of, or directly derived from, the target require special caution because high predictive importance may reflect construction rather than useful external signal.
4. **Identifier leakage:** client, page, URL, or other identifiers should not act as predictive features.
5. **Split leakage:** observations from the same client should not cross the grouped train/test boundary.

The audit is intended to establish whether the Week-5 feature set is suitable for this validation exercise. Any feature that directly reveals the target or future outcome should be removed rather than treated as a useful predictive signal.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Original claim

The Random Forest model improves ranking performance and can identify the best pages to refresh.

### Safer claim

The Week-5 evaluation **observed** an NDCG of **0.4629** for the Random Forest compared with **0.4391** for the Week-4 baseline. This is a **measured directional improvement** on the original evaluation setup.

After applying an honest grouped validation split, I use the resulting score to assess how stable that improvement is across unseen clients. The model should therefore be treated as **decision-support for prioritizing pages for review**, rather than as proof that a particular page will improve after refresh.

The evidence supports ranking candidate pages for investigation; it does not establish a causal effect of the recommended action or guarantee future search performance.


In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.